# EfficientNetB0 para PlantVillage

Implementacion reproducible de EfficientNetB0 como modelo de comparacion frente al baseline MobileNetV2 y el candidato MobileNetV3. Mantiene las mismas 38 clases de PlantVillage, el mismo split 80/10/10 y el mismo protocolo de exportacion TFLite para garantizar comparabilidad directa.

**Decision de arquitectura:** Se usa `EfficientNetB0` como primera variante por ser la mas liviana de la familia y comparable en tamano con MobileNetV3 Small. Si la accuracy no es suficiente se puede escalar a `B3` ajustando `MODEL_VARIANT`.

**Preprocessing:** EfficientNetV1 (B0-B7) aplica su propio rescaling interno y espera entrada en el rango `[0, 255]`. A diferencia de MobileNetV3, NO se usa `include_preprocessing=False` ni normalizacion manual a `[-1, 1]`. El pipeline solo redimensiona y convierte a `float32` — EfficientNet normaliza internamente.

## Configuracion

In [1]:
from pathlib import Path
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from sklearn.metrics import classification_report, confusion_matrix

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_DIR       = Path('/Users/carolinachavez/convolutional-neuronal-network/EfficientNet')
MOBILENET_DIR     = Path('/Users/carolinachavez/convolutional-neuronal-network/MobileNet')
ANDROID_ASSETS_DIR = Path('/Users/carolinachavez/Documents/Leaf-disease-applicaton/Leafdisease/app/src/main/assets')
CONTEXT_DOC       = Path('/Users/carolinachavez/convolutional-neuronal-network/context/model_efficientNet.md')

IMG_SIZE   = (224, 224)   # B0 nativo es 224; cambiar a (300,300) si MODEL_VARIANT='B3'
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE
NUM_CLASSES_EXPECTED = 38

# --- Variante del modelo ---
# 'B0' : ~5.3M params, 224x224, ~29 MB keras  -> recomendado para comparacion movil
# 'B3' : ~12M params, 300x300, ~48 MB keras   -> mayor accuracy, mayor tamano
MODEL_VARIANT  = 'B0'
MODEL_NAME     = f'efficientnet{MODEL_VARIANT.lower()}_v1'

# --- Modo de entrenamiento ---
# 'smoke_test' : corrida rapida para validar pipeline (pocos datos, pocas epocas)
# 'full'       : corrida completa para resultados finales
TRAINING_MODE = 'full'

SMOKE_TRAIN_EXAMPLES = 2048
SMOKE_VAL_EXAMPLES   = 512
SMOKE_TEST_EXAMPLES  = 512

HEAD_EPOCHS       = 3  if TRAINING_MODE == 'smoke_test' else 10
FINE_TUNE_EPOCHS  = 0  if TRAINING_MODE == 'smoke_test' else 10
HEAD_LR           = 1e-3
FINE_TUNE_LR      = 1e-5
FINE_TUNE_LAST_N_LAYERS = 20   # EfficientNetB0 tiene menos bloques que V3Large

RUN_TRAINING       = True
RUN_FINE_TUNING    = TRAINING_MODE == 'full'
RUN_EXPORT         = TRAINING_MODE == 'full'
RUN_DATASET_PROFILE = True

print('Python executable:', os.sys.executable)
print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('Training mode:', TRAINING_MODE)
print('Model variant: EfficientNet', MODEL_VARIANT)
print('IMG_SIZE:', IMG_SIZE)
print(tf.config.list_physical_devices('GPU'))

Python executable: /opt/anaconda3/envs/learning_ai/bin/python
TensorFlow: 2.20.0
Keras: 3.14.0
Training mode: full
Model variant: EfficientNet B0
IMG_SIZE: (224, 224)
[]


## Carga del dataset y preparacion

Se usa `tensorflow_datasets/plant_village` con el mismo split 80/10/10 y seed `123` que MobileNetV2 y MobileNetV3 para garantizar comparabilidad directa.

**Diferencia de preprocessing respecto a MobileNet:** EfficientNetV1 NO acepta el parametro `include_preprocessing` y aplica su propio rescaling interno. El pipeline entrega imagenes en `[0, 255]` como `float32` y el backbone se encarga del resto. Si en el futuro se integra este modelo en Android, el preprocesamiento de la app debera adaptarse a `[0, 255]` en lugar de `[-1, 1]`.

In [2]:
def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    # EfficientNetV1 aplica su propio rescaling interno: NO normalizar a [-1,1]
    # El backbone espera pixeles en [0, 255] como float32
    return image, label

raw_ds, info = tfds.load('plant_village', split='train', as_supervised=True, with_info=True)
class_names  = info.features['label'].names
num_classes  = info.features['label'].num_classes
total_examples = info.splits['train'].num_examples

assert num_classes == NUM_CLASSES_EXPECTED, f'Expected 38 classes, got {num_classes}'

train_size = int(total_examples * 0.80)
val_size   = int(total_examples * 0.10)
test_size  = total_examples - train_size - val_size

shuffled_ds = raw_ds.shuffle(total_examples, seed=SEED, reshuffle_each_iteration=False)
train_raw = shuffled_ds.take(train_size)
val_raw   = shuffled_ds.skip(train_size).take(val_size)
test_raw  = shuffled_ds.skip(train_size + val_size)

if TRAINING_MODE == 'smoke_test':
    train_raw = train_raw.take(SMOKE_TRAIN_EXAMPLES)
    val_raw   = val_raw.take(SMOKE_VAL_EXAMPLES)
    test_raw  = test_raw.take(SMOKE_TEST_EXAMPLES)

train_ds = train_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_ds   = val_raw.map(preprocess,   num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds  = test_raw.map(preprocess,  num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

effective_train_size = train_size
effective_val_size   = val_size
effective_test_size  = test_size

print(f'Clases: {num_classes}')
print(f'Train:  {effective_train_size}')
print(f'Val:    {effective_val_size}')
print(f'Test:   {effective_test_size}')


Clases: 38
Train:  43442
Val:    5430
Test:   5431


In [3]:
# Guardar labels en el directorio del proyecto y verificar alineacion con Android
labels_path = PROJECT_DIR / f'labels_efficientnet{MODEL_VARIANT.lower()}.txt'
labels_path.write_text('\n'.join(class_names) + '\n')

android_labels_path = ANDROID_ASSETS_DIR / 'labels.txt'
if android_labels_path.exists():
    android_labels = [l.strip() for l in android_labels_path.read_text().splitlines() if l.strip()]
    labels_match_android = android_labels == class_names
else:
    labels_match_android = False

print('Labels guardadas:', labels_path)
print('Labels coinciden con Android asset:', labels_match_android)

Labels guardadas: /Users/carolinachavez/convolutional-neuronal-network/EfficientNet/labels_efficientnetb0.txt
Labels coinciden con Android asset: True


## Grafica de distribucion de clases

In [ ]:
if RUN_DATASET_PROFILE:
    counts = np.zeros(num_classes, dtype=np.int64)
    for _, labels in raw_ds.batch(2048):
        counts += np.bincount(labels.numpy(), minlength=num_classes)

    dist_df = pd.DataFrame({'class_name': class_names, 'count': counts}).sort_values('count', ascending=False)
    plt.figure(figsize=(14, 8))
    sns.barplot(data=dist_df, x='count', y='class_name', color='#2196F3')
    plt.title('PlantVillage - distribucion de clases')
    plt.xlabel('Cantidad de imagenes')
    plt.ylabel('Clase')
    plt.tight_layout()
    plt.savefig(PROJECT_DIR / 'dataset_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Clases con menos de 500 imagenes:', (dist_df['count'] < 500).sum())

## Referencia de modelos existentes

Se registran los tamaños TFLite de MobileNetV2 y MobileNetV3 para tener el contexto de comparacion antes de entrenar.

In [4]:
def file_size_mb(path):
    path = Path(path)
    return round(path.stat().st_size / (1024 * 1024), 4) if path.exists() else None

reference_models = [
    {
        'model': 'MobileNetV2 FP16 (baseline Android)',
        'tflite_path': MOBILENET_DIR / 'saved_models' / 'mobilenetv2_v3' / 'mobilenetv2_v3_fp16.tflite',
        'test_accuracy': None,
    },
    {
        'model': 'MobileNetV3 Small FP16 (candidato)',
        'tflite_path': MOBILENET_DIR / 'saved_models' / 'mobilenetv3_small_v1_fp16.tflite',
        'test_accuracy': 0.9856,
    },
]

ref_df = pd.DataFrame([{
    'model': m['model'],
    'tflite_size_mb': file_size_mb(m['tflite_path']),
    'test_accuracy': m['test_accuracy'],
} for m in reference_models])

display(ref_df)

,model,tflite_size_mb,test_accuracy
0,MobileNetV2 FP16 (baseline Android),4.3494,NaN
1,MobileNetV3 Small FP16 (candidato),1.9089,0.9856


## Construccion del modelo EfficientNet

El backbone se carga con `include_top=False` y `weights='imagenet'`. EfficientNetV1 NO acepta `include_preprocessing` — el rescaling de `[0, 255]` a la distribucion interna lo aplica el propio backbone.

La cabeza agrega `GlobalAveragePooling2D`, `BatchNormalization`, `Dropout(0.3)` y `Dense(38, softmax)`.

**Por que BatchNormalization en la cabeza:** EfficientNet es sensible a cambios de distribucion durante transfer learning. Agregar una BN despues del pooling estabiliza el entrenamiento de la cabeza antes del fine-tuning.

In [5]:
def build_efficientnet(num_classes: int, variant: str = 'B0'):
    """Construye EfficientNet con cabeza de clasificacion para PlantVillage.

    Variantes soportadas: B0 (224x224), B3 (300x300), B4 (380x380).
    EfficientNetV1 NO acepta include_preprocessing: el backbone aplica su
    propio rescaling interno. El pipeline entrega pixeles en [0, 255] float32.
    """
    backbone_map = {
        'B0': tf.keras.applications.EfficientNetB0,
        'B3': tf.keras.applications.EfficientNetB3,
        'B4': tf.keras.applications.EfficientNetB4,
    }
    if variant not in backbone_map:
        raise ValueError(f'Variante no soportada: {variant}. Opciones: {list(backbone_map.keys())}')

    backbone = backbone_map[variant](
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
        # include_preprocessing NO existe en EfficientNetV1 —
        # el backbone aplica su propio rescaling internamente
    )
    backbone.trainable = False

    inputs  = keras.Input(shape=(*IMG_SIZE, 3), name='image')
    x       = backbone(inputs, training=False)
    x       = keras.layers.GlobalAveragePooling2D(name='gap')(x)
    x       = keras.layers.BatchNormalization(name='head_bn')(x)
    x       = keras.layers.Dropout(0.3, name='dropout')(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    model = keras.Model(inputs, outputs, name=f'efficientnet{variant.lower()}_plantvillage')
    return model, backbone


model, backbone = build_efficientnet(num_classes, variant=MODEL_VARIANT)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=HEAD_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary(show_trainable=True)


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "efficientnetb0_plantvillage"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ image (InputLayer)          │ (None, 224, 224, 3)   │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ efficientnetb0 (Functional) │ (None, 7, 7, 1280)    │  4,049,571 │   N   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ gap                         │ (None, 1280)          │          0 │   -   │
│ (GlobalAveragePooling2D)    │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ head_bn                     │ (None, 1280)          │      5,120 │   Y   │
│ (BatchNormalization)        │                       │            │       │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ dropout (Dropout)           │ (None, 1280)          │          0 │   -   │
├─────────────────────────────┼───────────────────────┼────────────┼───────┤
│ predictions (Dense)         │ (None, 38)            │     48,678 │   Y   │
└─────────────────────────────┴───────────────────────┴────────────┴───────┘

 Total params: 4,103,369 (15.65 MB)

 Trainable params: 51,238 (200.15 KB)

 Non-trainable params: 4,052,131 (15.46 MB)

## Entrenamiento de la cabeza

Primera fase: backbone congelado, solo se entrena la cabeza de clasificacion. Esto adapta las features preentrenadas de ImageNet a las 38 clases de PlantVillage con menor costo computacional y menor riesgo de sobreajuste.

In [ ]:
checkpoint_path   = PROJECT_DIR / f'{MODEL_NAME}_best.keras'
training_log_path = PROJECT_DIR / f'{MODEL_NAME}_training_log.csv'

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor='val_accuracy', mode='max',
        save_best_only=True, verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', mode='max',
        patience=4, restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.3,
        patience=2, min_lr=1e-7, verbose=1,
    ),
    keras.callbacks.CSVLogger(str(training_log_path)),
]

history_head      = None
history_finetune  = None

if RUN_TRAINING:
    print(f'== Fase 1: entrenamiento de cabeza ({HEAD_EPOCHS} epocas) ==')
    history_head = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS,
        callbacks=callbacks,
    )
else:
    print('RUN_TRAINING=False, saltando entrenamiento.')

== Fase 1: entrenamiento de cabeza (10 epocas) ==
Epoch 1/10


## Fine-tuning controlado

Segunda fase (solo en `TRAINING_MODE = 'full'`): se descongelan las ultimas `FINE_TUNE_LAST_N_LAYERS` capas del backbone con learning rate bajo. Las capas `BatchNormalization` se mantienen congeladas para estabilidad durante el fine-tuning.

In [ ]:
if RUN_TRAINING and RUN_FINE_TUNING:
    print(f'== Fase 2: fine-tuning ultimas {FINE_TUNE_LAST_N_LAYERS} capas ({FINE_TUNE_EPOCHS} epocas) ==')
    backbone.trainable = True
    for layer in backbone.layers[:-FINE_TUNE_LAST_N_LAYERS]:
        layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False

    trainable_count = sum(1 for l in backbone.layers if l.trainable)
    print(f'Capas del backbone entrenables: {trainable_count} / {len(backbone.layers)}')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    history_finetune = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS + FINE_TUNE_EPOCHS,
        initial_epoch=history_head.epoch[-1] + 1 if history_head else 0,
        callbacks=callbacks,
    )
else:
    print('Fine-tuning desactivado (smoke_test o RUN_TRAINING=False).')

## Graficas de entrenamiento

In [ ]:
def history_to_frame(*histories):
    frames, offset = [], 0
    for name, hist in histories:
        if hist is None:
            continue
        frame = pd.DataFrame(hist.history)
        frame['epoch'] = np.arange(offset, offset + len(frame)) + 1
        frame['phase'] = name
        frames.append(frame)
        offset += len(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

history_df = history_to_frame(('head', history_head), ('fine_tune', history_finetune))

if not history_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for col, ax, title in [
        (['accuracy', 'val_accuracy'], axes[0], 'Accuracy'),
        (['loss',     'val_loss'],     axes[1], 'Loss'),
    ]:
        for c in col:
            if c in history_df.columns:
                sns.lineplot(data=history_df, x='epoch', y=c, hue='phase', ax=ax, label=c)
        ax.set_title(f'EfficientNet{MODEL_VARIANT} - {title}')
        ax.set_xlabel('Epoca')
        ax.legend()
    plt.tight_layout()
    plt.savefig(PROJECT_DIR / 'training_graph.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Sin historial de entrenamiento para graficar.')

## Evaluacion y matriz de confusion

In [ ]:
if checkpoint_path.exists():
    eval_model = keras.models.load_model(checkpoint_path)
    print('Cargado desde checkpoint:', checkpoint_path)
else:
    eval_model = model
    print('Checkpoint no encontrado; usando modelo en memoria.')

test_loss, test_accuracy = eval_model.evaluate(test_ds, verbose=1)
print(f'Test loss:     {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')

y_true, y_pred, y_conf = [], [], []
for images, labels in test_ds:
    probs = eval_model.predict(images, verbose=0)
    preds = np.argmax(probs, axis=1)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(preds.tolist())
    y_conf.extend(probs.max(axis=1).tolist())

print('\nReporte por clase:')
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f'EfficientNet{MODEL_VARIANT} - Matriz de confusion PlantVillage')
plt.xlabel('Prediccion')
plt.ylabel('Real')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(PROJECT_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Errores con mayor confianza (los mas peligrosos para la app)
errors = [
    {'true_label': class_names[t], 'predicted_label': class_names[p], 'confidence': c}
    for t, p, c in zip(y_true, y_pred, y_conf) if t != p
]
errors_df = pd.DataFrame(errors)
if not errors_df.empty:
    print(f'Total de errores: {len(errors_df)} / {len(y_true)} ({len(errors_df)/len(y_true)*100:.1f}%)')
    display(errors_df.sort_values('confidence', ascending=False).head(25))
else:
    print('Sin errores en el conjunto de prueba.')

## Exportacion a SavedModel y TFLite

Se generan dos versiones TFLite:
- `fp32`: sin cuantizacion, precision maxima, mayor tamano
- `fp16`: cuantizacion float16, menor tamano (~50%), misma precision en la mayoria de dispositivos Android

La exportacion solo corre en `TRAINING_MODE = 'full'`.

In [ ]:
saved_model_dir  = PROJECT_DIR / 'saved_models' / MODEL_NAME
fp32_tflite_path = PROJECT_DIR / 'saved_models' / f'{MODEL_NAME}_fp32.tflite'
fp16_tflite_path = PROJECT_DIR / 'saved_models' / f'{MODEL_NAME}_fp16.tflite'

export_results = {}

if RUN_EXPORT:
    saved_model_dir.mkdir(parents=True, exist_ok=True)
    eval_model.export(str(saved_model_dir))
    print('SavedModel exportado:', saved_model_dir)

    # FP32
    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
    tflite_fp32 = converter.convert()
    fp32_tflite_path.write_bytes(tflite_fp32)

    # FP16
    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    tflite_fp16 = converter.convert()
    fp16_tflite_path.write_bytes(tflite_fp16)

    export_results = {
        'fp32_size_mb': file_size_mb(fp32_tflite_path),
        'fp16_size_mb': file_size_mb(fp16_tflite_path),
    }
    print('TFLite FP32:', export_results['fp32_size_mb'], 'MB ->', fp32_tflite_path)
    print('TFLite FP16:', export_results['fp16_size_mb'], 'MB ->', fp16_tflite_path)
else:
    print('Exportacion desactivada (smoke_test). Cambiar TRAINING_MODE="full" para exportar.')

## Benchmark local TFLite

Estima latencia en CPU local. No reemplaza la medicion en dispositivo Android real, pero permite detectar modelos demasiado lentos antes de integrar.

In [ ]:
import warnings

def benchmark_tflite(tflite_path, dataset, warmup=3, runs=25):
    tflite_path = Path(tflite_path)
    if not tflite_path.exists():
        return {'path': str(tflite_path), 'error': 'archivo no encontrado'}

    try:
        from ai_edge_litert.interpreter import Interpreter as LiteInterpreter
    except ImportError:
        warnings.filterwarnings('ignore', message='.*tf.lite.Interpreter is deprecated.*')
        LiteInterpreter = tf.lite.Interpreter

    interp = LiteInterpreter(model_path=str(tflite_path))
    interp.allocate_tensors()
    input_details  = interp.get_input_details()[0]
    output_details = interp.get_output_details()[0]

    sample_images, _ = next(iter(dataset.unbatch().batch(1)))
    sample = sample_images.numpy().astype(input_details['dtype'])

    timings_ms = []
    for i in range(warmup + runs):
        interp.set_tensor(input_details['index'], sample)
        t0 = time.perf_counter()
        interp.invoke()
        t1 = time.perf_counter()
        if i >= warmup:
            timings_ms.append((t1 - t0) * 1000)

    return {
        'path': str(tflite_path),
        'size_mb': round(tflite_path.stat().st_size / (1024**2), 4),
        'avg_ms':    round(np.mean(timings_ms), 2),
        'median_ms': round(np.median(timings_ms), 2),
        'p95_ms':    round(np.percentile(timings_ms, 95), 2),
        'std_ms':    round(np.std(timings_ms), 2),
    }

benchmarks = []
for path in [fp16_tflite_path, fp32_tflite_path]:
    result = benchmark_tflite(path, test_ds)
    benchmarks.append(result)
    print(result)

## Tabla comparativa de modelos

In [ ]:
mobilenetv3_fp16 = MOBILENET_DIR / 'saved_models' / 'mobilenetv3_small_v1_fp16.tflite'
mobilenetv2_fp16 = MOBILENET_DIR / 'saved_models' / 'mobilenetv2_v3' / 'mobilenetv2_v3_fp16.tflite'

comparison_rows = [
    {
        'model': 'MobileNetV2 FP16 (baseline Android)',
        'test_accuracy': None,
        'tflite_size_mb': file_size_mb(mobilenetv2_fp16),
        'avg_ms_local': None,
        'notas': 'Baseline actual de la app',
    },
    {
        'model': 'MobileNetV3 Small FP16',
        'test_accuracy': 0.9856,
        'tflite_size_mb': file_size_mb(mobilenetv3_fp16),
        'avg_ms_local': None,
        'notas': 'Candidato principal',
    },
    {
        'model': f'EfficientNet{MODEL_VARIANT} FP16',
        'test_accuracy': test_accuracy,
        'tflite_size_mb': file_size_mb(fp16_tflite_path),
        'avg_ms_local': next((b.get('avg_ms') for b in benchmarks if 'fp16' in b.get('path', '')), None),
        'notas': 'Este notebook',
    },
]

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

## Registro en contexto

Agrega un resumen de esta corrida al archivo de contexto `model_efficientNet.md`.

In [ ]:
def append_run_to_context():
    timestamp    = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    fp16_size    = file_size_mb(fp16_tflite_path)
    bench_fp16   = next((b for b in benchmarks if 'fp16' in b.get('path', '')), {})

    lines = [
        '',
        f'## Corrida EfficientNet{MODEL_VARIANT} - {timestamp}',
        '',
        f'- Notebook: `{PROJECT_DIR / "EfficientNetB0.ipynb"}`',
        f'- Modelo: `EfficientNet{MODEL_VARIANT}` con pesos ImageNet, `include_preprocessing=False`',
        f'- Training mode: `{TRAINING_MODE}`',
        f'- Dataset: PlantVillage, 38 clases, split 80/10/10, seed `{SEED}`',
        f'- Effective split sizes: train `{effective_train_size}`, validacion `{effective_val_size}`, test `{effective_test_size}`',
        f'- Test accuracy: `{test_accuracy:.4f}`',
        f'- Test loss: `{test_loss:.4f}`',
        f'- TFLite FP16 size MB: `{fp16_size}`',
        f'- Latencia local avg (ms): `{bench_fp16.get("avg_ms", "N/A")}`',
        f'- Latencia local P95 (ms): `{bench_fp16.get("p95_ms", "N/A")}`',
    ]

    CONTEXT_DOC.parent.mkdir(parents=True, exist_ok=True)
    with open(CONTEXT_DOC, 'a') as f:
        f.write('\n'.join(lines) + '\n')
    print('Contexto actualizado:', CONTEXT_DOC)

if TRAINING_MODE == 'full':
    append_run_to_context()
else:
    print('smoke_test: contexto no actualizado. Ejecutar con TRAINING_MODE="full" para registrar.')